In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [3]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [4]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [5]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [6]:

words = ["king", "computer", "city", "happy"]

for w in words:
    print(f"\nTop similar words for '{w}':")
    similar_words = glove_wiki_vectors.most_similar(w, topn=5)
    for word, score in similar_words:
        print(f"{word} ({score:.4f})")


Top similar words for 'king':
prince (0.7682)
queen (0.7508)
son (0.7021)
brother (0.6986)
monarch (0.6978)

Top similar words for 'computer':
computers (0.8752)
software (0.8373)
technology (0.7642)
pc (0.7366)
hardware (0.7290)

Top similar words for 'city':
town (0.8264)
cities (0.7764)
where (0.7548)
area (0.7458)
downtown (0.7438)

Top similar words for 'happy':
'm (0.8413)
feel (0.8133)
're (0.8048)
i (0.7938)
'll (0.7916)


In [7]:
...

Ellipsis

In [8]:
...

Ellipsis

In [9]:
...

Ellipsis

In [10]:
...

Ellipsis

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [11]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [12]:
for w1, w2 in word_pairs:
    similarity = glove_wiki_vectors.similarity(w1, w2)
    print(f"Similarity between '{w1}' and '{w2}': {similarity:.4f}")

Similarity between 'coast' and 'shore': 0.7000
Similarity between 'clothes' and 'closet': 0.5463
Similarity between 'old' and 'new': 0.6432
Similarity between 'smart' and 'intelligent': 0.7553
Similarity between 'dog' and 'cat': 0.8798
Similarity between 'tree' and 'lawyer': 0.0767


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [13]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [14]:
for word in test_words:
    if word in glove_wiki_vectors:
        print(f"{word}: In vocabulary")
    else:
        print(f"{word}: Not in vocabulary")

covididiot: Not in vocabulary
fomo: Not in vocabulary
frenemies: In vocabulary
anthropause: Not in vocabulary
photobomb: Not in vocabulary
selfie: Not in vocabulary
pxg: Not in vocabulary
pacg: Not in vocabulary
cct: In vocabulary
escc: In vocabulary


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [15]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [16]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [17]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283238)

In [18]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.35182396)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [19]:
analogy("man", "pilot", "woman")

man : pilot :: woman : ?


,Analogy word,Score
0,pilots,0.646806
1,flight,0.617965
2,helicopter,0.616516
3,crew,0.604449
4,plane,0.591008
5,landing,0.565227
6,nurse,0.564288
7,astronaut,0.564032
8,crash,0.562628
9,airplane,0.559773


In [20]:
analogy("man", "boss", "woman")

man : boss :: woman : ?


,Analogy word,Score
0,bosses,0.649429
1,girlfriend,0.639793
2,boyfriend,0.623138
3,colleague,0.604826
4,lover,0.593473
5,husband,0.590123
6,ex,0.575455
7,friend,0.565926
8,wife,0.555912
9,housekeeper,0.527948


In [21]:
print(glove_wiki_vectors.similarity("woman", "family"))
print(glove_wiki_vectors.similarity("man", "family"))

0.5514649
0.58080524


In [22]:
print(glove_wiki_vectors.similarity("rich", "success"))
print(glove_wiki_vectors.similarity("poor", "success"))

0.37711856
0.4904465


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

The results show that embeddings contain some associations, but they are not always consistent with common stereotypes. 

For example, the "pilot" analogy mainly returned aviation-related terms rather than a clear gendered role. The "boss" example included some gender-related words, but not strongly. Also, "man" was slightly more similar to "family" than "woman", and "poor" was more similar to "success" than "rich", which contradicts expectations.

This suggests that bias in embeddings can be subtle and inconsistent, so results should be interpreted carefully.

In practice, even small biases can be harmful. For example, in hiring systems, embeddings could influence how resumes are matched to jobs, potentially leading to unfair outcomes.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [23]:
from sklearn.datasets import fetch_20newsgroups

In [24]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

,text,target,target_name
0,"You know, I was reading 18 U.S.C. 922 and some...",6,talk.politics.guns
1,\n\n\nIt's not a bad question: I don't have an...,1,comp.graphics
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
...,...,...,...
4558,Hi Everyone ::\n\nI am looking for some soft...,1,comp.graphics
4559,Archive-name: x-faq/part3\nLast-modified: 1993...,2,comp.windows.x
4560,"\nThat's nice, but it doesn't answer the quest...",6,talk.politics.guns
4561,"Hi,\n I just got myself a Gateway 4DX-33V ...",2,comp.windows.x


In [25]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [26]:
import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [27]:
import re
def preprocess(text):

    text = text.lower()
    

    text = re.sub(r"\n+", " ", text)
    

    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"\d+", " NUM ", text)
    

    text = re.sub(r"[^a-z\s]", " ", text)
    

    text = re.sub(r"\s+", " ", text).strip()
    

    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        lemma = token.lemma_
        if len(lemma) > 2:
            tokens.append(lemma)
    
    return " ".join(tokens)


df["text_pp"] = df["text"].apply(preprocess)


df[["text", "text_pp"]].head()

,text,text_pp
0,"You know, I was reading 18 U.S.C. 922 and some...",know read sence wonder help provide paragraph ...
1,\n\n\nIt's not a bad question: I don't have an...,bad question don ref list algorithm think bit ...
2,"\nActuallay I don't, but on the other hand I d...",actuallay don hand don support idea have newsg...
3,"The following problem is really bugging me,\na...",follow problem bug appreciate help create wind...
4,\n\n This is the latest from UPI \n\n For...,late upi foreign ministry spokesman ferhat ata...


In [28]:
...

Ellipsis

In [29]:
...

Ellipsis

In [30]:
df.iloc[2:6]

,text,target,target_name,text_pp
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics,actuallay don hand don support idea have newsg...
3,"The following problem is really bugging me,\na...",2,comp.windows.x,follow problem bug appreciate help create wind...
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast,late upi foreign ministry spokesman ferhat ata...
5,"Hi,\n I'd like to subscribe to Leadership Ma...",5,soc.religion.christian,like subscribe leadership magazine wonder disk...


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

In [31]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

# vectorize preprocessed text
vectorizer = CountVectorizer(max_df=0.95, min_df=5)
X = vectorizer.fit_transform(df["text_pp"])

# try a few topic numbers
topic_nums = [6, 8, 10, 12]
results = []

for k in topic_nums:
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42
    )
    lda.fit(X)
    results.append({
        "n_topics": k,
        "perplexity": lda.perplexity(X)
    })

results_df = pd.DataFrame(results)
results_df

,n_topics,perplexity
0,6,1859.137852
1,8,1859.690483
2,10,1853.990277
3,12,1757.525576


In [32]:
# choose a reasonable number of topics
best_k = 8

lda_model = LatentDirichletAllocation(
    n_components=best_k,
    random_state=42
)
lda_model.fit(X)

feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

Topic 1: state, file, israel, israeli, bill, gun, right, law, government, national
Topic 2: gun, period, firearm, win, handgun, crime, rate, weapon, det, van
Topic 3: god, people, jesus, jews, say, law, turkish, time, know, right
Topic 4: say, people, know, come, don, kill, think, time, tell, didn
Topic 5: think, don, believe, people, know, god, thing, christian, church, good
Topic 6: file, program, window, image, use, entry, available, display, run, include
Topic 7: point, know, thank, don, lib, computer, like, problem, book, post
Topic 8: game, team, year, play, player, win, good, season, think, get


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

In [33]:
feature_names = vectorizer.get_feature_names_out()

topic_labels = {}

for topic_idx, topic in enumerate(lda_model.components_):
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    
    print(f"Topic {topic_idx}: {', '.join(top_words)}")
    
    # Assign labels manually based on top words
    if any(w in top_words for w in ["god", "jesus", "bible", "christian"]):
        label = "Religion"
    elif any(w in top_words for w in ["game", "team", "season", "player"]):
        label = "Sports"
    elif any(w in top_words for w in ["file", "image", "graphics", "software"]):
        label = "Computer Graphics"
    elif any(w in top_words for w in ["gun", "weapon", "firearm"]):
        label = "Guns"
    elif any(w in top_words for w in ["government", "law", "state", "policy"]):
        label = "Politics"
    else:
        label = "General/Mixed"
    
    topic_labels[topic_idx] = label
    
    print(f"--> Label: {label}\n")

Topic 0: state, file, israel, israeli, bill, gun, right, law, government, national
--> Label: Computer Graphics

Topic 1: gun, period, firearm, win, handgun, crime, rate, weapon, det, van
--> Label: Guns

Topic 2: god, people, jesus, jews, say, law, turkish, time, know, right
--> Label: Religion

Topic 3: say, people, know, come, don, kill, think, time, tell, didn
--> Label: General/Mixed

Topic 4: think, don, believe, people, know, god, thing, christian, church, good
--> Label: Religion

Topic 5: file, program, window, image, use, entry, available, display, run, include
--> Label: Computer Graphics

Topic 6: point, know, thank, don, lib, computer, like, problem, book, post
--> Label: General/Mixed

Topic 7: game, team, year, play, player, win, good, season, think, get
--> Label: Sports



<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [34]:

doc_topic_dist = lda_model.transform(X)
doc_topics_df = pd.DataFrame(doc_topic_dist)

doc_topics_df["dominant_topic"] = doc_topics_df.idxmax(axis=1)


doc_topics_df.head()

,0,1,2,3,4,5,6,7,dominant_topic
0,0.198659,0.002318,0.413367,0.344230,0.002319,0.002318,0.034474,0.002317,2
1,0.001787,0.001786,0.001787,0.001787,0.001787,0.001788,0.987490,0.001787,6
2,0.002051,0.002051,0.002051,0.002052,0.559420,0.125149,0.305174,0.002051,4
3,0.005001,0.005004,0.005008,0.005016,0.165537,0.804429,0.005004,0.005002,5
4,0.041179,0.003125,0.630906,0.312282,0.003127,0.003128,0.003126,0.003127,2


<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. Content-based filtering recommends items based on their similarity to items the user has liked before. It uses features of items to match user preferences.

2. One negative consequence is the creation of filter bubbles, where users are only exposed to similar content and miss diverse perspectives. Another issue is bias reinforcement, where existing preferences or societal biases are amplified over time.

3. Transfer learning in NLP is the process of taking a model trained on a large dataset and fine-tuning it on a smaller, specific task. This allows models to perform well even with limited data.

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)